# Lab 5 — Hosted Agent (Basic)

A **hosted agent** is *your own container* (built with the [Agent Framework](https://github.com/microsoft/agent-framework)) deployed to Foundry. Foundry runs it, gives it a dedicated endpoint and its own managed identity, and manages sessions/streaming for the Responses protocol.

In this lab you will:

1. Scaffold the agent's container source (`main.py`, `agent.yaml`, `Dockerfile`, ...).
2. Deploy it to Foundry with the Azure Developer CLI (`azd`).
3. Invoke the deployed agent over the Responses protocol.

> **Extra prerequisites for this lab:** Docker running locally, and the azd AI agent extension:
>
> ```bash
> azd extension install azure.ai.agents
> azd auth login
> ```


## 1. Scaffold the agent source

The cells below write a complete hosted-agent project into `./lab5_agent/`. The agent uses `FoundryChatClient` for the model and `ResponsesHostServer` to expose the OpenAI-compatible `/responses` endpoint.


In [ ]:
import os
os.makedirs("lab5_agent/src/labs-hosted-agent-basic", exist_ok=True)
print("Created lab5_agent/src/labs-hosted-agent-basic")


In [ ]:
%%writefile lab5_agent/src/labs-hosted-agent-basic/main.py
# Copyright (c) Microsoft. All rights reserved.
import os

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import ResponsesHostServer
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()


def main():
    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        credential=DefaultAzureCredential(),
    )
    agent = Agent(
        client=client,
        instructions="You are a friendly assistant. Keep your answers brief.",
        default_options={"store": False},
    )
    server = ResponsesHostServer(agent)
    server.run()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile lab5_agent/src/labs-hosted-agent-basic/agent.yaml
# yaml-language-server: $schema=https://raw.githubusercontent.com/microsoft/AgentSchema/refs/heads/main/schemas/v1.0/ContainerAgent.yaml
kind: hosted
name: labs-hosted-agent-basic
description: |
  A basic Agent Framework agent hosted by Foundry.
protocols:
  - protocol: responses
    version: 1.0.0
resources:
  cpu: "1"
  memory: 2Gi
environment_variables:
  - name: AZURE_AI_MODEL_DEPLOYMENT_NAME
    value: gpt-4.1


In [ ]:
%%writefile lab5_agent/src/labs-hosted-agent-basic/Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY . user_agent/
WORKDIR /app/user_agent
RUN if [ -f requirements.txt ]; then pip install -r requirements.txt; fi
EXPOSE 8088
CMD ["python", "main.py"]


In [ ]:
%%writefile lab5_agent/src/labs-hosted-agent-basic/requirements.txt
agent-framework
agent-framework-foundry-hosting


In [ ]:
%%writefile lab5_agent/azure.yaml
# yaml-language-server: $schema=https://raw.githubusercontent.com/Azure/azure-dev/main/schemas/v1.0/azure.yaml.json
name: labs-hosted-agent-basic
requiredVersions:
  extensions:
    azure.ai.agents: '>=0.1.0-preview'
services:
  labs-hosted-agent-basic:
    project: src/labs-hosted-agent-basic
    host: azure.ai.agent
    language: docker
    docker:
      remoteBuild: true
    config:
      container:
        resources:
          cpu: "1"
          memory: 2Gi
      startupCommand: python main.py


## 2. Deploy with azd

Deployment builds the container (remote ACR build), creates the agent, and registers its environment variables. Run these in a **terminal** from the `lab5_agent` folder (Docker must be running):

```bash
cd lab5_agent
azd ai agent init            # bind to your EXISTING Foundry project + pick model gpt-4.1
azd deploy
```

When prompted during `azd ai agent init`, choose your existing project (do **not** create a new one) and select a model that is actually deployed in your project (e.g. `gpt-4.1`).

> **Tip:** a cosmetic `postdeploy` 404 can appear if the azd service name differs from the `agent.yaml` `name`. It is safe to ignore — the agent still deploys.

You can run the deploy from the notebook too (it can take a few minutes and needs Docker):


In [ ]:
# Optional: deploy from the notebook. Requires Docker running and `azd ai agent init` already done.
# Uncomment to run.
# !cd lab5_agent && azd deploy


## 3. Invoke the deployed agent

Once the agent version is **active**, invoke it over the Responses protocol. The endpoint path is:

```
{project_endpoint}/agents/{name}/endpoint/protocols/openai/responses?api-version=v1
```

In [ ]:
import os
import requests
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv()

AGENT_NAME = "labs-hosted-agent-basic"
project_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].rstrip("/")

token = DefaultAzureCredential().get_token("https://ai.azure.com/.default").token
url = f"{project_endpoint}/agents/{AGENT_NAME}/endpoint/protocols/openai/responses?api-version=v1"

resp = requests.post(
    url,
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"input": "Say hello in one short sentence.", "store": False},
    timeout=120,
)
data = resp.json()
print("status:", data.get("status"))
for item in data.get("output", []):
    for part in item.get("content", []):
        if part.get("type") == "output_text":
            print(part["text"])

You now have a hosted agent running your own container code. In **Lab 6** you'll add **RAG over Azure AI Search** so the agent grounds its answers in your documents.
